In [1]:
%pip install langchain langchain-core langchain-community langchain-groq \
             langchain-openai langchain-anthropic \
             langchain-text-splitters pandas tabulate --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sqlite3
import pandas as pd
from getpass import getpass
from pathlib import Path
from tabulate import tabulate

# LangChain components
from langchain_core.prompts       import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables     import RunnablePassthrough
from langchain_anthropic          import ChatAnthropic
from langchain_groq               import ChatGroq
from langchain_openai             import ChatOpenAI

print("All imports successful.")

All imports successful.


In [3]:
# ── CHOOSE YOUR PROVIDER ─────────────────────────────────────────────────────
PROVIDER = "groq"     # "claude" | "groq" | "openai"
# ─────────────────────────────────────────────────────────────────────────────

if PROVIDER == "claude":
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")
    llm = ChatAnthropic(model="claude-sonnet-4-6", temperature=0)
    print("LLM: Anthropic — claude-sonnet-4-6")

elif PROVIDER == "groq":
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    print("LLM: Groq — llama-3.3-70b-versatile (free tier)")

elif PROVIDER == "openai":
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    print("LLM: OpenAI — gpt-4o-mini")

else:
    raise ValueError(f"Unknown PROVIDER: '{PROVIDER}'. Choose claude, groq, or openai.")

print("\nConfiguration complete. Ready to build.")

LLM: Groq — llama-3.3-70b-versatile (free tier)

Configuration complete. Ready to build.


In [4]:
# ── Quick demo — run AFTER completing the full notebook ──────────────────────
# The cells below build the database and the bot step by step.
# This cell assumes you have run all cells first.

DEMO_DB = "retail.db"

_demo_questions = [
    "How many orders were placed in total?",
    "Which product category has the highest total revenue?",
    "Who are the top 3 customers by total spend?",
    "What is the weather in Bangalore today?",     # out of scope
]

print("=" * 65)
print("DEMO — Text-to-SQL Bot")
print("Database: retail.db (orders, products, customers)")
print("=" * 65)

try:
    for q in _demo_questions:
        result = bot.ask(q)
        print(f"\nQ: {q}")
        print(f"SQL: {result['sql']}")
        if result["data"] is not None:
            print(result["data"].to_string(index=False))
        else:
            print(result["answer"])
        print("-" * 65)
except NameError:
    print("Run all cells below first, then come back to this demo.")

DEMO — Text-to-SQL Bot
Database: retail.db (orders, products, customers)
Run all cells below first, then come back to this demo.


In [5]:
import sqlite3
import random
from datetime import datetime, timedelta

DB_PATH = "retail.db"

# Remove existing DB to start fresh
if Path(DB_PATH).exists():
    Path(DB_PATH).unlink()

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# ── Create tables ─────────────────────────────────────────────────────────────
cur.executescript("""
    CREATE TABLE customers (
        id        INTEGER PRIMARY KEY AUTOINCREMENT,
        name      TEXT    NOT NULL,
        city      TEXT    NOT NULL,
        join_date TEXT    NOT NULL
    );

    CREATE TABLE products (
        id       INTEGER PRIMARY KEY AUTOINCREMENT,
        name     TEXT    NOT NULL,
        category TEXT    NOT NULL,
        price    REAL    NOT NULL
    );

    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_id INTEGER NOT NULL REFERENCES customers(id),
        product_id  INTEGER NOT NULL REFERENCES products(id),
        quantity    INTEGER NOT NULL,
        order_date  TEXT    NOT NULL,
        total       REAL    NOT NULL
    );
""")

# ── Seed data ─────────────────────────────────────────────────────────────────
customers = [
    ("Arjun Sharma",   "Bangalore"), ("Priya Nair",    "Mumbai"),
    ("Ravi Kumar",     "Delhi"),     ("Ananya Singh",  "Chennai"),
    ("Vikram Patel",   "Pune"),      ("Meena Iyer",    "Hyderabad"),
    ("Rohit Verma",    "Kolkata"),   ("Sita Reddy",    "Bangalore"),
    ("Karan Mehta",    "Mumbai"),    ("Deepa Krishnan","Delhi"),
]

products = [
    ("Laptop Pro 15",     "Electronics",  75000.0),
    ("Wireless Earbuds",  "Electronics",   3500.0),
    ("Python Crash Course","Books",         799.0),
    ("Standing Desk",     "Furniture",    22000.0),
    ("Mechanical Keyboard","Electronics",   6500.0),
    ("Office Chair",      "Furniture",    18500.0),
    ("Data Science Handbook","Books",       1299.0),
    ("USB-C Hub",         "Electronics",   2200.0),
    ("Notebook Set",      "Stationery",     450.0),
    ("Monitor 27inch",    "Electronics",  32000.0),
]

# Insert customers and products
base_date = datetime(2024, 1, 1)
for i, (name, city) in enumerate(customers):
    jd = (base_date + timedelta(days=i*30)).strftime('%Y-%m-%d')
    cur.execute('INSERT INTO customers (name, city, join_date) VALUES (?,?,?)', (name, city, jd))

for name, category, price in products:
    cur.execute('INSERT INTO products (name, category, price) VALUES (?,?,?)', (name, category, price))

# Generate 120 realistic orders
random.seed(42)
for _ in range(120):
    cust_id  = random.randint(1, len(customers))
    prod_id  = random.randint(1, len(products))
    qty      = random.randint(1, 3)
    price    = products[prod_id - 1][2]
    total    = round(price * qty, 2)
    days_ago = random.randint(0, 365)
    odate    = (datetime(2024, 1, 1) + timedelta(days=days_ago)).strftime('%Y-%m-%d')
    cur.execute(
        'INSERT INTO orders (customer_id, product_id, quantity, order_date, total) VALUES (?,?,?,?,?)',
        (cust_id, prod_id, qty, odate, total)
    )

conn.commit()
conn.close()

# Verify
conn = sqlite3.connect(DB_PATH)
for table in ['customers', 'products', 'orders']:
    count = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f"  {table:12s}: {count} rows")
conn.close()
print(f"\nDatabase ready: {DB_PATH}")

  customers   : 10 rows
  products    : 10 rows
  orders      : 120 rows

Database ready: retail.db


In [6]:
def extract_schema(db_path: str, sample_rows: int = 3) -> str:
    """
    Extract table schema and sample rows from a SQLite database.
    Returns a formatted string ready for injection into an LLM prompt.
    """
    conn   = sqlite3.connect(db_path)
    cursor = conn.cursor()
    schema_parts = []

    # Get all table names
    tables = cursor.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()

    for (table_name,) in tables:
        # Get column info via PRAGMA
        columns = cursor.execute(
            f'PRAGMA table_info({table_name})'
        ).fetchall()

        col_defs = ', '.join(
            f"{col[1]}({col[2]})" for col in columns
        )

        # Fetch sample rows so LLM understands the data format
        samples = cursor.execute(
            f'SELECT * FROM {table_name} LIMIT {sample_rows}'
        ).fetchall()

        sample_str = '\n    '.join(
            ' | '.join(str(v) for v in row) for row in samples
        ) if samples else '  (no rows yet)'

        schema_parts.append(
            f'Table: {table_name}\n'
            f'Columns: {col_defs}\n'
            f'Sample rows:\n    {sample_str}'
        )

    conn.close()
    return '\n\n'.join(schema_parts)

# Extract and display the schema
schema = extract_schema(DB_PATH)
print("Extracted schema:")
print("=" * 60)
print(schema)

Extracted schema:
Table: customers
Columns: id(INTEGER), name(TEXT), city(TEXT), join_date(TEXT)
Sample rows:
    1 | Arjun Sharma | Bangalore | 2024-01-01
    2 | Priya Nair | Mumbai | 2024-01-31
    3 | Ravi Kumar | Delhi | 2024-03-01

Table: orders
Columns: id(INTEGER), customer_id(INTEGER), product_id(INTEGER), quantity(INTEGER), order_date(TEXT), total(REAL)
Sample rows:
    1 | 2 | 1 | 3 | 2024-05-20 | 225000.0
    2 | 4 | 4 | 1 | 2024-02-22 | 22000.0
    3 | 9 | 2 | 3 | 2024-08-04 | 10500.0

Table: products
Columns: id(INTEGER), name(TEXT), category(TEXT), price(REAL)
Sample rows:
    1 | Laptop Pro 15 | Electronics | 75000.0
    2 | Wireless Earbuds | Electronics | 3500.0
    3 | Python Crash Course | Books | 799.0

Table: sqlite_sequence
Columns: name(), seq()
Sample rows:
    customers | 10
    products | 10
    orders | 120


In [7]:
SQL_PROMPT_TEMPLATE = """\
You are an expert SQL assistant for a SQLite database.
Use ONLY the tables and columns defined in the schema below.

RULES:
1. Return ONLY the raw SQL query — no explanation, no markdown, no backticks.
2. Use only columns and tables that exist in the schema.
3. If the question cannot be answered from this database, respond with:
   NOT_SQL: <brief reason why>
4. For aggregations, always use meaningful column aliases.
5. Limit results to 20 rows unless the question asks for all.

DATABASE SCHEMA:
──────────────────────────────────────
{schema}
──────────────────────────────────────

Question: {question}

SQL Query:"""

sql_prompt = PromptTemplate(
    template        = SQL_PROMPT_TEMPLATE,
    input_variables = ["schema", "question"],
)

# LCEL chain: prompt → LLM → string output
sql_chain = sql_prompt | llm | StrOutputParser()

print("SQL prompt and chain ready.")
print(f"Schema length: {len(schema)} chars — fits comfortably in any context window.")

SQL prompt and chain ready.
Schema length: 828 chars — fits comfortably in any context window.


In [8]:
def execute_sql(sql: str, db_path: str) -> dict:
    """
    Execute a SELECT SQL query against the database.
    Returns a dict with keys: data (DataFrame), error (str), rows (int).
    """
    sql_clean = sql.strip()

    # Safety check — only allow SELECT statements
    if not sql_clean.upper().startswith('SELECT'):
        return {
            'data'  : None,
            'error' : f'Only SELECT queries are allowed. Got: {sql_clean[:40]}...',
            'rows'  : 0
        }

    try:
        conn = sqlite3.connect(db_path)
        df   = pd.read_sql_query(sql_clean, conn)
        conn.close()
        return {
            'data'  : df,
            'error' : None,
            'rows'  : len(df)
        }
    except sqlite3.Error as e:
        return {
            'data'  : None,
            'error' : str(e),
            'rows'  : 0
        }

# Quick sanity check
test = execute_sql('SELECT COUNT(*) as total_orders FROM orders', DB_PATH)
print("Sanity check — direct query:")
print(test['data'].to_string(index=False))

Sanity check — direct query:
 total_orders
          120


In [9]:
def ask(question: str, db_path: str = DB_PATH) -> dict:
    """
    Ask a plain-English question about the database.
    Returns a dict with: sql, data (DataFrame), answer, rows, error.
    """
    # Refresh schema on every call — reflects live DB changes
    live_schema = extract_schema(db_path)

    # Generate SQL from the question
    generated = sql_chain.invoke({
        'schema'  : live_schema,
        'question': question,
    }).strip()

    # Check for out-of-scope response
    if generated.upper().startswith('NOT_SQL:'):
        reason = generated.split(':', 1)[1].strip()
        return {
            'sql'   : None,
            'data'  : None,
            'answer': reason,
            'rows'  : 0,
            'error' : None,
        }

    # Execute the generated SQL
    result = execute_sql(generated, db_path)

    return {
        'sql'   : generated,
        'data'  : result['data'],
        'answer': None,
        'rows'  : result['rows'],
        'error' : result['error'],
    }

# First live test
result = ask("How many orders were placed in total?")
print(f"SQL: {result['sql']}")
print(f"Result:")
print(result['data'].to_string(index=False))

SQL: SELECT COUNT(id) AS total_orders FROM orders
Result:
 total_orders
          120


In [10]:
test_questions = [
    "How many orders were placed in total?",
    "Which product category has the highest total revenue?",
    "Who are the top 3 customers by total spend?",
    "What is the weather in Bangalore today?",        # out of scope
]

print("=" * 65)
for q in test_questions:
    result = ask(q)
    print(f"\nQ: {q}")

    if result['sql']:
        print(f"SQL: {result['sql']}")

    if result['error']:
        print(f"Error: {result['error']}")
    elif result['data'] is not None:
        print(result['data'].to_string(index=False))
    else:
        print(f"Answer: {result['answer']}")

    print("-" * 65)


Q: How many orders were placed in total?
SQL: SELECT COUNT(id) AS total_orders FROM orders
 total_orders
          120
-----------------------------------------------------------------

Q: Which product category has the highest total revenue?
SQL: SELECT category, SUM(total) AS total_revenue FROM products p JOIN orders o ON p.id = o.product_id GROUP BY category ORDER BY total_revenue DESC LIMIT 1
   category  total_revenue
Electronics      3287500.0
-----------------------------------------------------------------

Q: Who are the top 3 customers by total spend?
SQL: SELECT c.name, SUM(o.total) AS Total_Spend
FROM customers c
JOIN orders o ON c.id = o.customer_id
GROUP BY c.name
ORDER BY Total_Spend DESC
LIMIT 3
        name  Total_Spend
  Priya Nair    1013594.0
 Karan Mehta     646493.0
Ananya Singh     485243.0
-----------------------------------------------------------------

Q: What is the weather in Bangalore today?
Answer: No weather data available in the database
--------------

In [11]:
class TextToSQL:
    """
    Production-ready Text-to-SQL bot for SQLite databases.
    Works on any SQLite database — no hardcoded schema.
    """

    _PROMPT = """\
You are an expert SQL assistant for a SQLite database.
Use ONLY the tables and columns defined in the schema below.

RULES:
1. Return ONLY the raw SQL query — no explanation, no markdown, no backticks.
2. Use only columns and tables that exist in the schema.
3. If the question cannot be answered from this database, respond with:
   NOT_SQL: <brief reason why>
4. For aggregations, always use meaningful column aliases.
5. Limit results to 20 rows unless the question asks for all.

DATABASE SCHEMA:
──────────────────────────────────────
{schema}
──────────────────────────────────────

Question: {question}

SQL Query:"""

    def __init__(self, db_path: str, provider: str = 'groq'):
        self.db_path  = db_path
        self.provider = provider
        self._init_llm()
        self._init_chain()

    def _init_llm(self):
        """Initialise LLM based on provider string."""
        if self.provider == 'claude':
            self.llm = ChatAnthropic(model='claude-sonnet-4-6', temperature=0)
        elif self.provider == 'groq':
            self.llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
        elif self.provider == 'openai':
            self.llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
        else:
            raise ValueError(f"Unknown provider: '{self.provider}'. Choose claude, groq, or openai.")

    def _init_chain(self):
        """Build the LCEL SQL generation chain."""
        prompt      = PromptTemplate(
            template        = self._PROMPT,
            input_variables = ['schema', 'question'],
        )
        self.chain  = prompt | self.llm | StrOutputParser()

    def ask(self, question: str) -> dict:
        """
        Ask a plain-English question about the database.
        Returns: {sql, data, answer, rows, error}
        """
        schema    = extract_schema(self.db_path)
        generated = self.chain.invoke({
            'schema'  : schema,
            'question': question,
        }).strip()

        if generated.upper().startswith('NOT_SQL:'):
            return {
                'sql': None, 'data': None,
                'answer': generated.split(':', 1)[1].strip(),
                'rows': 0, 'error': None,
            }

        result = execute_sql(generated, self.db_path)
        return {
            'sql'   : generated,
            'data'  : result['data'],
            'answer': None,
            'rows'  : result['rows'],
            'error' : result['error'],
        }

    def describe(self) -> None:
        """Print a readable summary of the database schema."""
        print(f"Database: {self.db_path}")
        print("=" * 50)
        conn   = sqlite3.connect(self.db_path)
        tables = conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        ).fetchall()
        for (t,) in tables:
            cols  = conn.execute(f'PRAGMA table_info({t})').fetchall()
            count = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
            print(f"\n  {t} ({count} rows)")
            for col in cols:
                print(f"    {col[1]:20s} {col[2]}")
        conn.close()

print("TextToSQL class defined.")

TextToSQL class defined.


In [12]:
# Initialise — change provider= to switch LLM
bot = TextToSQL(db_path=DB_PATH, provider=PROVIDER)

# Show what the bot knows about the database
bot.describe()

print("\n" + "=" * 65)
print("End-to-end test — TextToSQL class")
print("=" * 65)

e2e_questions = [
    "How many customers are from Bangalore?",
    "What is the average order total?",
    "Which product has been ordered the most times?",
    "List the top 5 orders by total value with customer names.",
    "What is the stock price of Infosys today?",      # out of scope
]

for q in e2e_questions:
    r = bot.ask(q)
    print(f"\nQ: {q}")
    if r['sql']:
        print(f"SQL: {r['sql']}")
    if r['error']:
        print(f"Error: {r['error']}")
    elif r['data'] is not None:
        print(tabulate(r['data'], headers='keys', tablefmt='simple', showindex=False))
    else:
        print(f"Answer: {r['answer']}")
    print("-" * 65)

Database: retail.db

  customers (10 rows)
    id                   INTEGER
    name                 TEXT
    city                 TEXT
    join_date            TEXT

  sqlite_sequence (3 rows)
    name                 
    seq                  

  products (10 rows)
    id                   INTEGER
    name                 TEXT
    category             TEXT
    price                REAL

  orders (120 rows)
    id                   INTEGER
    customer_id          INTEGER
    product_id           INTEGER
    quantity             INTEGER
    order_date           TEXT
    total                REAL

End-to-end test — TextToSQL class

Q: How many customers are from Bangalore?
SQL: SELECT COUNT(id) AS number_of_customers FROM customers WHERE city = 'Bangalore'
  number_of_customers
---------------------
                    2
-----------------------------------------------------------------

Q: What is the average order total?
SQL: SELECT AVG(total) AS average_order_total FROM orders
  aver